In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("reconciliation_table", "oh_apm_stg.tmp.ctl_reconciliation_log_Prv")
dbutils.widgets.text("error_table", "oh_apm_stg.vendor_extracts.error_load_report_log_cpc_Stg_Prv")
dbutils.widgets.text("s3_path", "s3://gia-stg-oh-ue1-data-raw/haven/inbound/VE_EDW/weekly")

In [0]:
reconciliation_table = dbutils.widgets.get("reconciliation_table")
error_table = dbutils.widgets.get("error_table")
s3_path = dbutils.widgets.get("s3_path")

# -*- coding: utf-8 -*-
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, LongType
import ast

# -------------------------------
# Utility: safe get task values
# -------------------------------
def safe_get(task_key, key, default):
    try:
        return dbutils.jobs.taskValues.get(taskKey=task_key, key=key, debugValue=default)
    except Exception:
        return default

# -------------------------------
# Coercion helpers (handle native or stringified repr)
# -------------------------------
def coerce_list(x, default=None):
    if default is None: default = []
    if isinstance(x, list): return x
    if isinstance(x, str):
        s = x.strip()
        if not s: return default
        try:
            v = ast.literal_eval(s)
            return v if isinstance(v, list) else default
        except Exception:
            return default
    return default

def coerce_dict(x, default=None):
    if default is None: default = {}
    if isinstance(x, dict): return x
    if isinstance(x, str):
        s = x.strip()
        if not s: return default
        try:
            v = ast.literal_eval(s)
            return v if isinstance(v, dict) else default
        except Exception:
            return default
    return default

def flatten_expected_names(seq):
    """
    gz_expected_info may be: list[str], list[tuple], or list[dict].
    Normalize to list[str] of file names.
    """
    out = []
    for item in seq:
        if isinstance(item, (list, tuple)) and len(item) >= 1:
            out.append(item[0])
        elif isinstance(item, dict) and "file_name" in item:
            out.append(item["file_name"])
        elif isinstance(item, str):
            out.append(item)
    return out

# -------------------------------
# Retrieve pre-validation outputs
# -------------------------------
gz_expected_info_raw = safe_get("Pre_validation", "gz_expected_info", [])
gz_file_paths_raw = safe_get("Pre_validation", "gz_file_paths", {})
missing_required_raw = safe_get("Pre_validation", "missing_required_gz_files", [])
ctl_received_ts = safe_get("Pre_validation", "ctl_received_ts", None)
start_load = safe_get("Pre_validation", "start_time", None)

# Get error table from widget
error_table = dbutils.widgets.get("error_table")

# -------------------------------
# Coerce & normalize
# -------------------------------
gz_expected_info = coerce_list(gz_expected_info_raw, default=[])
gz_file_paths = coerce_dict(gz_file_paths_raw, default={})
missing_required_gz_files = coerce_list(missing_required_raw, default=[])
expected_names = flatten_expected_names(gz_expected_info)
missing_required_set = set(missing_required_gz_files)

# -------------------------------
# Default timestamps if not set
# -------------------------------
if not start_load:
    start_load = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
end_load = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
if not ctl_received_ts:
    ctl_received_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# -------------------------------
# Prepare error records (two-phase)
# -------------------------------
error_records = []
phase = None

# Schema
error_schema = StructType([
    StructField("File_Name", StringType(), True),
    StructField("Date_Received", StringType(), True),   # empty when missing
    StructField("Start_Load_Date", StringType(), True),
    StructField("End_Load_Date", StringType(), True),
    StructField("Row_Number", LongType(), True),
    StructField("Error_Description", StringType(), True)
])

# ---------- Phase 1: CTL mismatch ----------
if missing_required_gz_files:
    phase = "CTL"

    # Build a common error message that references the missing file(s), like your example
    if len(missing_required_gz_files) == 1:
        common_err = f"MISSING {missing_required_gz_files[0]} FILE."
    else:
        common_err = "MISSING " + ", ".join(missing_required_gz_files) + " FILES."

    if expected_names:
        # Write one row per expected file.
        # Date_Received = ctl_received_ts for those NOT missing; blank for the missing one(s).
        for file_name in expected_names:
            date_received = "" if file_name in missing_required_set else ctl_received_ts
            error_records.append((file_name, date_received, start_load, end_load, 0, common_err))
    else:
        # Fallback: if no expected list provided, at least write rows for the missing ones
        for file_name in missing_required_gz_files:
            error_records.append((file_name, "", start_load, end_load, 0, f"MISSING {file_name} FILE."))

# ---------- Phase 2: S3 check (only if CTL passed) ----------
else:
    phase = "S3"
    keys = set(gz_file_paths.keys())
    missing_in_s3 = [name for name in expected_names if name not in keys]
    for file_name in missing_in_s3:
        error_records.append((file_name, "", start_load, end_load, 0, f"MISSING {file_name} FILE."))

# -------------------------------
# Write or no-op depending on errors
# -------------------------------
if error_records:
    error_df = spark.createDataFrame(error_records, schema=error_schema)
    try:
        error_df.write.mode("append").saveAsTable(error_table)
        print(f"✅ Error report written to {error_table} (phase={phase}, count={len(error_records)})")
        display(error_df)
        dbutils.jobs.taskValues.set(key="Copy_to_APMTable", value=True)
    except Exception as e:
        raise Exception(f"❌ Failed to write error table: {e}")
else:
    print("✅ No missing files detected.")
    dbutils.jobs.taskValues.set(key="Copy_to_APMTable", value=False)

# -------------------------------
# Optional: debug
# -------------------------------
print(f"🔹 Phase: {phase}")
print(f"🔹 Start Load: {start_load}")
print(f"🔹 End Load: {end_load}")
print(f"🔹 Date Received: {ctl_received_ts}")
print(f"🔹 missing_required_gz_files (raw): {missing_required_raw}")
print(f"🔹 gz_expected_info (raw): {gz_expected_info_raw}")
print(f"🔹 gz_file_paths (raw): {gz_file_paths_raw}")
print(f"🔹 Missing Required Files in CTL (coerced): {missing_required_gz_files}")
print(f"🔹 Expected Names (coerced): {expected_names}")
print(f"🔹 gz_file_paths keys (count): {len(gz_file_paths)}")
if phase == "S3":
    missing_gz_files = [name for name in expected_names if name not in gz_file_paths]
    print(f"🔹 Missing GZ Files in S3: {missing_gz_files}")
print(f"🔹 Total Error Records: {len(error_records)}")


In [0]:
# -*- coding: utf-8 -*-
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, LongType
import ast, re

# -------------------------------
# Utilities
# -------------------------------
def safe_get(task_key, key, default):
    try:
        return dbutils.jobs.taskValues.get(taskKey=task_key, key=key, debugValue=default)
    except Exception:
        return default

def try_get_widget(name, default=None):
    try:
        return dbutils.widgets.get(name)
    except Exception:
        return default

def coerce_list(x, default=None):
    if default is None: default = []
    if isinstance(x, list): return x
    if isinstance(x, str):
        s = x.strip()
        if not s: return default
        try:
            v = ast.literal_eval(s)
            return v if isinstance(v, list) else default
        except Exception:
            return default
    return default

def coerce_dict(x, default=None):
    if default is None: default = {}
    if isinstance(x, dict): return x
    if isinstance(x, str):
        s = x.strip()
        if not s: return default
        try:
            v = ast.literal_eval(s)
            return v if isinstance(v, dict) else default
        except Exception:
            return default
    return default

def normalize_expected_names(obj):
    """Normalize gz_expected_info into list[str]. Handles list/tuple/dict/str."""
    names = []

    def add(name):
        if name is None:
            return
        n = str(name).strip()
        if n:
            names.append(n)

    if isinstance(obj, (list, tuple, set)):
        for item in obj:
            if isinstance(item, (list, tuple)) and len(item) >= 1:
                first = item[0]
                if isinstance(first, dict):
                    for key in ("file_name", "File_Name", "filename", "name"):
                        if key in first:
                            add(first[key]); break
                else:
                    add(first)
            elif isinstance(item, dict):
                picked = False
                for key in ("file_name", "File_Name", "filename", "name"):
                    if key in item:
                        add(item[key]); picked = True; break
                if not picked and len(item) == 1:
                    add(next(iter(item.keys())))
            elif isinstance(item, str):
                add(item)
            else:
                add(item)
    elif isinstance(obj, dict):
        for k in obj.keys():
            add(k)
    elif isinstance(obj, str):
        s = obj.strip()
        if s:
            try:
                v = ast.literal_eval(s)
                return normalize_expected_names(v)
            except Exception:
                toks = re.findall(r'[\w.\-]+\.gz', s)
                if toks:
                    names.extend(toks)
                else:
                    for part in re.split(r'[,\s]+', s):
                        if part:
                            add(part)
    else:
        add(obj)

    seen = set()
    uniq = []
    for n in names:
        if n not in seen:
            seen.add(n)
            uniq.append(n)
    return uniq

# --- CTL parsing helpers ---
def normalize_ts(s):
    """Normalize dates like 'YYYYMMDDHHMMSS' or 'YYYY-MM-DD HH:MM:SS' to 'YYYY-MM-DD HH:MM:SS'."""
    if not s:
        return None
    s = s.strip()
    # Already formatted?
    m = re.match(r'^(\d{4})-(\d{2})-(\d{2}) T:(\d{2}):(\d{2})$', s)
    if m:
        return f"{m.group(1)}-{m.group(2)}-{m.group(3)} {m.group(4)}:{m.group(5)}:{m.group(6)}"
    # Compact: YYYYMMDDHHMMSS or YYYYMMDD HHMMSS
    m2 = re.match(r'^(\d{4})(\d{2})(\d{2})[ T]?(\d{2})(\d{2})(\d{2})$', s)
    if m2:
        return f"{m2.group(1)}-{m2.group(2)}-{m2.group(3)} {m2.group(4)}:{m2.group(5)}:{m2.group(6)}"
    # Date-only: YYYYMMDD
    m3 = re.match(r'^(\d{4})(\d{2})(\d{2})$', s)
    if m3:
        return f"{m3.group(1)}-{m3.group(2)}-{m3.group(3)} 00:00:00"
    return s  # fallback raw

def read_ctl_lines(path, sample_only=False, sample_limit=10000):
    """Read CTL as text lines with spark or dbutils.fs.head fallback."""
    if not path:
        return []
    try:
        df = spark.read.text(path)
        if sample_only:
            rows = df.limit(sample_limit).collect()
        else:
            rows = df.collect()
        return [r['value'] for r in rows]
    except Exception:
        try:
            # read up to ~1MB
            content = dbutils.fs.head(path, 1024 * 1024)
            return content.splitlines()
        except Exception:
            return []

def parse_ctl_for_names_and_dates(lines):
    """
    Extract (*.gz) file names and optional timestamps from CTL lines.
    Heuristics: look for tokens matching *.gz and timestamps on the same line.
    """
    gz_pat = re.compile(r'([\w.\-]+\.gz)')
    # common timestamp patterns
    ts_pats = [
        re.compile(r'\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}'),
        re.compile(r'\d{8}[ T]?\d{6}'),
        re.compile(r'\d{8}')  # date-only as last resort
    ]
    names = []
    date_map = {}  # file -> ts
    for line in lines or []:
        gz_matches = gz_pat.findall(line)
        if not gz_matches:
            continue
        ts = None
        for p in ts_pats:
            m = p.search(line)
            if m:
                ts = normalize_ts(m.group(0))
                break
        for nm in gz_matches:
            if nm not in names:
                names.append(nm)
            if ts and nm not in date_map:
                date_map[nm] = ts
    return names, date_map

# -------------------------------
# Retrieve pre-validation outputs
# -------------------------------
gz_expected_info_raw = safe_get("Pre_validation", "gz_expected_info", [])
gz_file_paths_raw = safe_get("Pre_validation", "gz_file_paths", {})
missing_required_raw = safe_get("Pre_validation", "missing_required_gz_files", [])
ctl_received_ts = safe_get("Pre_validation", "ctl_received_ts", None)
start_load = safe_get("Pre_validation", "start_time", None)

# Inputs from job/widget
error_table = try_get_widget("error_table")
ctl_path = try_get_widget("ctl_path") or safe_get("Pre_validation", "ctl_path", None)

# -------------------------------
# Coerce & normalize
# -------------------------------
gz_expected_info = coerce_list(gz_expected_info_raw, default=[])
gz_file_paths = coerce_dict(gz_file_paths_raw, default={})
missing_required_gz_files = coerce_list(missing_required_raw, default=[])

expected_names = normalize_expected_names(gz_expected_info)
missing_required_set = set(missing_required_gz_files)

# -------------------------------
# Default timestamps if not set
# -------------------------------
if not start_load:
    start_load = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
end_load = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
if not ctl_received_ts:
    ctl_received_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# -------------------------------
# Prepare error records (two-phase)
# -------------------------------
error_records = []
phase = None

# Schema
error_schema = StructType([
    StructField("File_Name", StringType(), True),
    StructField("Date_Received", StringType(), True),   # empty when missing
    StructField("Start_Load_Date", StringType(), True),
    StructField("End_Load_Date", StringType(), True),
    StructField("Row_Number", LongType(), True),
    StructField("Error_Description", StringType(), True)
])

# ---------- Phase 1: CTL mismatch ----------
if missing_required_gz_files:
    phase = "CTL"

    # If we didn't get expected_names, derive them from the CTL file locally
    ctl_names, ctl_dates_map = [], {}
    if not expected_names and ctl_path:
        lines = read_ctl_lines(ctl_path)
        ctl_names, ctl_dates_map = parse_ctl_for_names_and_dates(lines)
        # Build expected list as union: CTL-present + missing list
        seen = set()
        expected_names = []
        for nm in ctl_names + list(missing_required_gz_files):
            if nm not in seen:
                seen.add(nm)
                expected_names.append(nm)

    # Compose the error message
    if len(missing_required_gz_files) == 1:
        common_err = f"MISSING {missing_required_gz_files[0]} FILE."
    else:
        common_err = "MISSING " + ", ".join(missing_required_gz_files) + " FILES."

    if expected_names:
        for file_name in expected_names:
            # Blank date if this file is one of the missing
            if file_name in missing_required_set:
                date_received = ""
            else:
                # Prefer per-file date parsed from CTL line, else fallback to ctl_received_ts
                date_received = ctl_dates_map.get(file_name, ctl_received_ts)
            error_records.append((file_name, date_received, start_load, end_load, 0, common_err))
    else:
        # Fallback: if still nothing, log at least the missing ones
        for file_name in missing_required_gz_files:
            error_records.append((file_name, "", start_load, end_load, 0, f"MISSING {file_name} FILE."))

# ---------- Phase 2: S3 check (only if CTL passed) ----------
else:
    phase = "S3"
    keys = set(gz_file_paths.keys())
    missing_in_s3 = [name for name in expected_names if name not in keys]
    for file_name in missing_in_s3:
        error_records.append((file_name, "", start_load, end_load, 0, f"MISSING {file_name} FILE."))

# -------------------------------
# Write or no-op depending on errors
# -------------------------------
if error_records:
    error_df = spark.createDataFrame(error_records, schema=error_schema)
    try:
        error_df.write.mode("append").saveAsTable(error_table)
        print(f"✅ Error report written to {error_table} (phase={phase}, count={len(error_records)})")
        display(error_df)
        dbutils.jobs.taskValues.set(key="Copy_to_APMTable", value=True)
    except Exception as e:
        raise Exception(f"❌ Failed to write error table: {e}")
else:
    print("✅ No missing files detected.")
    dbutils.jobs.taskValues.set(key="Copy_to_APMTable", value=False)

# -------------------------------
# Debug (helpful if still only 1 row is written)
# -------------------------------
print(f"🔹 Phase: {phase}")
print(f"🔹 Start Load: {start_load}")
print(f"🔹 End Load: {end_load}")
print(f"🔹 Date Received: {ctl_received_ts}")
print(f"🔹 ctl_path: {ctl_path}")
print(f"🔹 missing_required_gz_files (raw): {missing_required_raw}")
print(f"🔹 gz_expected_info (raw): {gz_expected_info_raw}")
print(f"🔹 Expected Names (final): {expected_names}")
if phase == "CTL" and ctl_path:
    print(f"🔹 Parsed CTL names count: {len(ctl_names)} (first 10: {ctl_names[:10]})")
    print(f"🔹 Parsed CTL dates sample: {list(dict(list(ctl_dates_map.items())[:5]).items())}")
if phase == "S3":
    print(f"🔹 gz_file_paths keys (count): {len(gz_file_paths)}")
    missing_gz_files = [name for name in expected_names if name not in gz_file_paths]
    print(f"🔹 Missing GZ Files in S3: {missing_gz_files}")
print(f"🔹 Total Error Records: {len(error_records)}")
